`py-g5ht-pipeline`

__NOTE__

You will need ~200GB of storage space locally for a single recording. After registration, you can delete ~150GB
- .nd2: 50GB  (~55GB if separate channel alignment recording taken)
- .h5: 9GB
- .tifs created from shear_correction, channel_alignment, bleach_correction: 47GB each     (shear_correction, channel_alignment can be deleted after running the full pipeline)
- .tifs created from warping and registering: 18GB each      (warped can be deleted after running the full pipeline)

__BATCH PROCESSING__:

This notebook is meant to process a single recording (except nose annotation, which happens for all unprocessed recordings). 

If you want to batch processs:

- set up `datasets.txt` such that unprocessed data are in the `# UNPROCESSED` section
- run `batch_pipeline_stage1.py`
- annotate nose positions by running the nose annotator (see `ANNOTATE NOTE POSITIONS` below)
- run `batch_pipeline_stage2.py`
- find and save a fixed frame and mask to register to (TODO)
- run `batch_pipeline_stage3.py`

## IMPORTS

In [2]:
import sys
import os
import importlib
from tqdm import tqdm
from pathlib import Path
from datetime import datetime
import shutil
import pandas as pd
import utils

## SPECIFY DATA TO PROCESS

In [3]:
data_dir = r'/store1/shared/g5ht/data' # this is a linux machine

# dataset (see datasets.txt)
dataset = 'date-20260112_strain-ISg5HT_condition-fedpatch_worm001.nd2'

NOISE_PTH = '/home/munib/code/g5ht-pipeline/noise/noise_111125.tif'

In [4]:
# directories and paths

INPUT_ND2 = dataset
date_str = INPUT_ND2.split('_')[0].split('-')[1]
local_dir = os.path.join(data_dir, date_str)
os.makedirs(local_dir, exist_ok=True)
INPUT_ND2_PTH = os.path.join(local_dir, INPUT_ND2)


OUT_DIR = utils.get_output_dir(INPUT_ND2_PTH)

STACK_LENGTH = 41 if 'immo' not in INPUT_ND2 else 122

# for recordings prior to roughly December 2025, we want to keep all but the last two z-slices, during which the piezo position is unstable
# after December 2025, we want to keep all but the first two z-slices, during which the piezo position is unstable at the beginning of the recording
date_obj = datetime.strptime(date_str, '%Y%m%d')
if date_obj < datetime(2025, 12, 1):
    z2keep =  (0,STACK_LENGTH-2) # tuple representing range of z-slices to keep, should keep all but the last two slices
else:
    z2keep =  (2,STACK_LENGTH) # tuple representing range of z-slices to keep, should keep all but the first two slices

# get noise stack and metadata from nd2
noise_stack = utils.get_noise_stack(NOISE_PTH, STACK_LENGTH)
num_frames, height, width, num_channels = utils.get_range_from_nd2(INPUT_ND2_PTH, stack_length=STACK_LENGTH) 
beads_alignment_file = utils.get_beads_alignment_file(INPUT_ND2_PTH)

## 1. SHEAR CORRECTION 

` conda activate g5ht-pipeline`

~ 1 hour for 1200 image stacks with 2 color channels, 41 z, 512 h, 512 w

- shear corrects each volume
  - depending on each exposure time, it can take roughly half a second between the first and last frames of a volume, so any movements need to be corrected for
- creates one `.tif` for each volume and stores it in the `shear_corrected` directory

In [ ]:
import shear_correct
_ = importlib.reload(sys.modules['shear_correct'])

start_index = "0"
end_index = str(num_frames-1)
# start_index = "515"
# end_index = str(num_frames-1)

ncpu = str(utils.get_optimal_cpu_count())

skip_shear_correction = False # if True, will just denoise and save as tif

# sys.argv = ["", nd2 file, start_frame, end_frame, noise_pth, stack_length, n_workers, num_frames, height, width, num_channels]
sys.argv = ["", INPUT_ND2_PTH, start_index, end_index, NOISE_PTH, STACK_LENGTH, ncpu, num_frames, height, width, num_channels, z2keep, skip_shear_correction]

# Call the main function
shear_correct.main()

## 2. CHANNEL ALIGNMENT

` conda activate g5ht-pipeline`

Takes around 30 minutes for 1200 image stacks with 2 color channels, 39 z, 512 h, 512 w

### 2a. GET MEDIAN CHANNEL ALIGNMENT PARAMETERS FROM ALL FRAMES

- If channel alignment file found, uses that, if not uses worm recording
- creates a `.txt` file for each volume that contains elastix channel registration parameters
- creates `chan_align_params.csv` and  `chan_align.txt`

In [ ]:
import get_channel_alignment
import median_channel_alignment
_ = importlib.reload(sys.modules['get_channel_alignment'])
_ = importlib.reload(sys.modules['median_channel_alignment'])

## set beads_alignment_file to None to use worm recording for channel alignment, even if beads file exists
# beads_alignment_file = None

start_index = "0"
end_index = str(num_frames-1)

ncpu = str(utils.get_optimal_cpu_count())


every_other = True # if True, will only use every other frame for channel alignment, which should be sufficient and will speed up the process by 2x

if beads_alignment_file is not None:
    align_with_beads = True
    every_other = False # if using beads for alignment, we want to use all frames because there's only 50
    num_frames_beads, _, _, _ = utils.get_range_from_nd2(beads_alignment_file, stack_length=STACK_LENGTH) 
    sys.argv = ["", beads_alignment_file, start_index, end_index, NOISE_PTH, STACK_LENGTH, ncpu, num_frames_beads, height, width, num_channels, every_other, align_with_beads]
else:
    align_with_beads = False
    sys.argv = ["", INPUT_ND2_PTH, start_index, end_index, NOISE_PTH, STACK_LENGTH, ncpu, num_frames, height, width, num_channels, every_other, align_with_beads]

# # Call the main function
get_channel_alignment.main()
median_channel_alignment.main()


### 2b. APPLY MEDIAN CHANNEL ALIGNMENT PARAMETERS

- ouputs aligned volumes in `channel_aligned` directory

In [ ]:
import apply_channel_alignment
_ = importlib.reload(sys.modules['apply_channel_alignment'])

start_index = "0"
end_index = str(num_frames-1)

ncpu = str(utils.get_optimal_cpu_count())

if beads_alignment_file is not None:
    align_with_beads = True
    num_frames_beads, _, _, _ = utils.get_range_from_nd2(beads_alignment_file, stack_length=STACK_LENGTH) 
    sys.argv = ["", INPUT_ND2_PTH, start_index, end_index, NOISE_PTH, STACK_LENGTH, ncpu, num_frames, height, width, num_channels, align_with_beads, beads_alignment_file]
else:
    align_with_beads = False
    sys.argv = ["", INPUT_ND2_PTH, start_index, end_index, NOISE_PTH, STACK_LENGTH, ncpu, num_frames, height, width, num_channels, align_with_beads]


# Call the main function
apply_channel_alignment.main()

### 2c. PLOT CHANNEL ALIGNMENT PARAMETER DISTRIBUTIONS

In [ ]:
import apply_channel_alignment
_ = importlib.reload(sys.modules['apply_channel_alignment'])

apply_channel_alignment.plot_channel_alignment_params(INPUT_ND2_PTH, beads_alignment_file)

## 3. BLEACH CORRECTION

Takes about 5 minutes

In [ ]:
import importlib
import os
import sys

import bleach_correct
_ = importlib.reload(sys.modules['bleach_correct'])


PTH = os.path.splitext(INPUT_ND2_PTH)[0]
REG_DIR = 'channel_aligned' # 'channel_aligned' or 'tif' 
channels = 1 # 0-gfp, 1-rfp
method = 'block' # 'block' or 'exponential'
mode = 'total' # 'total' or 'median'
output_dir = os.path.join(PTH, 'bleach_corrected')

bleach_correct.correct_bleaching(os.path.join(PTH,REG_DIR), output_dir=output_dir, channels=channels, method=method, fbc=0.04, intensity_mode=mode)

## 4. MIP

` conda activate g5ht-pipeline`

- outputs `means.png`, `focus.png`, `mip.tif`, and `mip.mp4`, `focus_check.csv`

##### TODO: 
- mip for xy, xz, zy
- mip for several slices

In [ ]:
import mip
_ = importlib.reload(sys.modules['mip'])

# command-line arguments
framerate = 8
tif_dir = 'bleach_corrected' # one of 'shear_corrected' 'channel_aligned' 'bleach_corrected'
# tif_dir = 'shear_corrected'
rmax = 850
gmax = 150
mp4_quality = 10
do_focus = True
sys.argv = ["", INPUT_ND2_PTH, tif_dir, STACK_LENGTH, num_frames, framerate, rmax, gmax, mp4_quality, do_focus]

# Call the main function
mip.main()

## 5 DRIFT ESTIMATION

` conda activate g5ht-pipeline`

- outputs  `z_selection.csv`, `z_selection_diagnostics.png`, `sharpness.csv`

Few minutes to run

TODO:
- use z selection going forward
- also use sharpness/focus (and other things) to determine good/bad frames

In [ ]:
import drift_estimation
_ = importlib.reload(sys.modules['drift_estimation'])

# command-line arguments
tif_dir = 'bleach_corrected' # one of 'shear_corrected' 'channel_aligned' 'bleach_corrected'

sys.argv = ["", INPUT_ND2_PTH, tif_dir, STACK_LENGTH, num_frames]

# Call the main function
drift_estimation.main()

## 6. SEGMENT

- outputs `label.tif`, contains segmented MIP for each volume

__on home pc__: 
`conda activate segment-torch`

Uses a separate conda environment from the rest of the pipeline. create it using:
`conda env create -f segment_torch.yml`

__on lab pc__: 
`conda activate torchcu129`

Uses a separate conda environment from the rest of the pipeline. create it following steps in:
`segment_torch_cu129_environment.yml`

__setup each time model weights change__
Need to set path to model weights as `CHECKPOINT` in `eval_torch.py`

In [ ]:
import segment_torch
_ = importlib.reload(sys.modules['segment_torch'])

mip_tif = 'mip_bleach_corrected'

MIP_PTH = os.path.join(os.path.splitext(INPUT_ND2_PTH)[0], f'{mip_tif}.tif')

DEVICE = 2

# command-line arguments
sys.argv = ["", MIP_PTH, DEVICE]

segment_torch.main()

## 7. SPLINE

`conda activate g5ht-pipeline`

- outputs `spline.json`, `spline.tif`, and `dilated.tif`

In [ ]:
import spline
_ = importlib.reload(sys.modules['spline'])

LABEL_PTH = MIP_PTH = os.path.join(os.path.splitext(INPUT_ND2_PTH)[0], 'label.tif')

# command-line arguments
sys.argv = ["", LABEL_PTH]

spline.main()

## ANNOTATE NOSE POSITIONS

Interactive tool to annotate nose (y,x) on the first frame (and optional constraint frames) for each unprocessed dataset.

- **Click** on the plot to set the nose position for the current frame
- **Slider** to scroll through frames
- **Save CSV** writes `orient_nose.csv` to the dataset directory
- **Next/Prev Dataset** navigates between datasets (auto-saves)
- **Delete Annot.** removes the annotation for the current frame

_install interactive matplotlib packages_: `uv pip install ipympl ipywidgets`

In [ ]:
%matplotlib widget
import importlib
import nose_annotator
_ = importlib.reload(nose_annotator)
from nose_annotator import NoseAnnotator

nd2_paths = utils.parse_datasets('/home/munib/code/g5ht-pipeline/datasets.txt', section='UNPROCESSED')
annotator = NoseAnnotator(nd2_paths)

## 8. ORIENT

`conda activate g5ht-pipeline`

- outputs `oriented.json`, `oriented.png`, `oriented_stack.tif`

- annotate nose positions using nose annotator, before running this code

In [ ]:
import orient
_ = importlib.reload(sys.modules['orient'])

SPLINE_PTH = MIP_PTH = os.path.join(os.path.splitext(INPUT_ND2_PTH)[0], 'spline.json')

# read nose position and constraints from orient_nose.csv
orient_csv = os.path.join(os.path.splitext(INPUT_ND2_PTH)[0], 'orient_nose.csv')
orient_df = pd.read_csv(orient_csv)

# frame 0 row gives the primary nose position
nose_row = orient_df[orient_df['frame'] == 0].iloc[0]
nose_y = int(nose_row['nose_y'])
nose_x = int(nose_row['nose_x'])

# remaining rows are constraint frames
constraint_rows = orient_df[orient_df['frame'] != 0].sort_values('frame')

sys.argv = ["", SPLINE_PTH, str(nose_y), str(nose_x)]

if len(constraint_rows) > 0:
    for _, row in constraint_rows.iterrows():
        sys.argv.extend([str(int(row['frame'])), str(int(row['nose_y'])), str(int(row['nose_x']))])

orient.main()

## 9. WARP

`conda activate g5ht-pipeline`

- ouputs: `warped/*.tif` and `masks/*.tif`

TODO: parallelize

takes 1-2 hours

In [ ]:
import warp
_ = importlib.reload(sys.modules['warp'])

PTH = os.path.splitext(INPUT_ND2_PTH)[0]

start_index = 0
end_index = num_frames

for i in tqdm(range(start_index, end_index)):
    # command-line arguments
    sys.argv = ["", PTH, i]

    warp.main()

## 9b. SELECT FIXED/REFERENCE FRAME

`conda activate g5ht-pipeline`

Ranks all warped frames by a composite score and copies top-10 candidates to `potential_fixed/`.

Scoring criteria (configurable weights):
- **Straightness** (0.25) — how straight the worm posture is (from oriented spline)
- **Sharpness** (0.25) — mean Laplacian variance across z-slices
- **RFP representativeness** (0.25) — ZNCC of frame's RFP MIP vs temporal-mean RFP MIP
- **Z-coverage** (0.15) — number of usable z-slices
- **Temporal bias** (0.10) — Gaussian prior favoring middle of recording

Outputs: `fixed_frame_candidates.csv`, `fixed_frame_montage.png`, `potential_fixed/`

In [5]:
import select_fixed
_ = importlib.reload(sys.modules['select_fixed'])

PTH = os.path.splitext(INPUT_ND2_PTH)[0]

candidates_df = select_fixed.main(PTH)

Dataset: /store1/shared/g5ht/data/20260112/date-20260112_strain-ISg5HT_condition-fedpatch_worm001
Candidates: 10 | Subsample: 10
Computing straightness...
  521 frames
Loading sharpness...
Loading z-coverage...
Computing mean RFP MIP (subsampled)...


Computing mean RFP MIP: 100%|██████████| 53/53 [00:00<00:00, 146.51it/s]


Computing per-frame RFP ZNCC...


Computing RFP ZNCC: 100%|██████████| 521/521 [00:23<00:00, 22.57it/s]



Candidates saved: /store1/shared/g5ht/data/20260112/date-20260112_strain-ISg5HT_condition-fedpatch_worm001/fixed_frame_candidates.csv

Top 10 candidates:
 rank  frame  composite_score  straightness   sharpness     zncc  n_usable  temporal_weight
    1    257         0.813205      0.974698 2802.201749 0.881578        24         0.999188
    2    442         0.784446      0.911798 3165.304736 0.877357        24         0.112535
    3    473         0.776447      0.936972 3259.136743 0.901461        22         0.050065
    4    255         0.763116      0.961228 2800.330181 0.883323        23         0.997996
    5    297         0.763043      0.912241 3036.555404 0.900696        22         0.915445
    6    228         0.759458      0.930461 2816.294371 0.922632        23         0.932354
    7    231         0.758757      0.914244 2893.845161 0.904169        23         0.943925
    8    316         0.757097      0.922521 2998.336486 0.911307        22         0.815250
    9    225     

### CONFIRM FIXED FRAME

Review `fixed_frame_montage.png` and `potential_fixed/`, then set `fixed_frame` below to your chosen frame index.

In [ ]:
# set this to your chosen frame index from the candidates
fixed_frame = int(candidates_df.iloc[0]['frame'])  # default: top-ranked candidate

PTH = os.path.splitext(INPUT_ND2_PTH)[0]
select_fixed.set_fixed_frame(PTH, fixed_frame)

## 10. REGISTER

`conda activate g5ht-pipeline`

__ALTERNATIVELY__: register using the wholistic registration algorithm, currently in MATLAB

TODO: parallelize / make faster

- pick a good representative fixed frame that you want to register everything to
  - copy it to the main output folder and name it `fixed_xxxx.tif`
  - copy the corresponding mask and name it `fixed_mask_xxxx.tif`

Takes around 5 hours for 1200 stacks

### MAKE COPIES OF FIXED FRAME AND MASK

In [4]:
fixed_frame = '0450'

PTH = os.path.splitext(INPUT_ND2_PTH)[0]
warped_dir = os.path.join(PTH, 'warped')
mask_dir = os.path.join(PTH, 'masks')

warped_fixed_frame_pth = os.path.join(warped_dir, f'{fixed_frame}.tif')
mask_fixed_frame_pth = os.path.join(mask_dir, f'{fixed_frame}.tif')

shutil.copy(warped_fixed_frame_pth, os.path.join(PTH, f'fixed_{fixed_frame}.tif'))
shutil.copy(mask_fixed_frame_pth, os.path.join(PTH, f'fixed_mask_{fixed_frame}.tif'))

'/store1/shared/g5ht/data/20260112/date-20260112_strain-ISg5HT_condition-fedpatch_worm001/fixed_mask_0450.tif'

### REGISTER

In [ ]:
import reg
_ = importlib.reload(sys.modules['reg'])

PTH = os.path.splitext(INPUT_ND2_PTH)[0]

start_index = 0
end_index = num_frames
zoom = 1

for i in tqdm(range(start_index, end_index)):
    # command-line arguments
    try:
        sys.argv = ["", PTH, i, str(zoom)]
        reg.main()
    except Exception as e:
        print(f"Error processing index {i}: {e}")   

  0%|          | 0/521 [00:00<?, ?it/s]

  0%|          | 2/521 [01:20<5:44:22, 39.81s/it]